# Module 6: Model Context Protocol (MCP)

**Day 4 — LangGraph Agents, Memory, HITL & MCP**

## What you will learn
- **What MCP is**: Anthropic's standard protocol for AI-tool connectivity
- **MultiServerMCPClient**: connect to multiple MCP servers at once
- **Tool discovery**: agent discovers tools at runtime, not hardcoded
- **Popular MCP servers**: filesystem, GitHub, Postgres, Brave Search, Databricks
- **Mock MCP tools**: test MCP patterns without installing servers

## MCP Mental Model
```
Your Agent  <->  MCP Client  <->  MCP Server (filesystem, github, postgres, ...)
```
MCP is to AI tools what REST is to web APIs — a standard interface everyone speaks.


In [ ]:
import sys
sys.path.insert(0, '../src')
print('Path configured.')

## 1. MCP Server Catalogue

Anthropic and the community maintain dozens of MCP servers. Your agent connects
to them and discovers their tools at runtime.

In [ ]:
from day4.mcp_integration import show_mcp_servers

servers = show_mcp_servers()
print(f'Available MCP servers: {len(servers)}')
print()

for server in servers:
    print(f'[{server["name"]}]')
    print(f'  Package:   {server["package"]}')
    print(f'  Tools:     {', '.join(server["tools"][:3])}')
    print(f'  Use case:  {server["use_case"]}')
    print()

## 2. MCP Setup Code

This is how you connect to real MCP servers using `langchain-mcp-adapters`.
The agent discovers available tools dynamically at connection time.

In [ ]:
from day4.mcp_integration import show_mcp_setup

print('MCP Setup Code:')
print(show_mcp_setup())

## 3. Mock MCP Tools

These mock tools simulate real MCP servers without needing Node.js or server installation.
The mock tools have the exact same interface as real MCP tools — swap for real ones in production.

In [ ]:
from day4.mcp_integration import mcp_filesystem_read, mcp_github_create_issue, mcp_databricks_run_sql

# Test mock tools
print('[filesystem] Reading /tmp/products.txt:')
print(' ', mcp_filesystem_read.invoke({'file_path': '/tmp/products.txt'}))

print()
print('[filesystem] Reading /tmp/config.json:')
print(' ', mcp_filesystem_read.invoke({'file_path': '/tmp/config.json'}))

print()
print('[github] Creating issue:')
result = mcp_github_create_issue.invoke({
    'title': 'AI agent hallucinating on edge cases',
    'body': 'The agent returns incorrect results for empty input',
    'repo': 'myorg/ai-platform'
})
print(' ', result)

print()
print('[databricks] Running SQL:')
print(' ', mcp_databricks_run_sql.invoke({'query': 'SELECT count(*) FROM sales_transactions WHERE date > "2024-01-01"'}))
print(' ', mcp_databricks_run_sql.invoke({'query': 'SELECT SUM(amount) as revenue FROM orders'}))

## 4. MCP Agent with Mock Tools

Build a complete LangGraph agent using MCP tools.

In [ ]:
from day4.mcp_integration import build_mcp_agent
from day4.tools_agents import MockLLMWithTools
from langchain_core.messages import HumanMessage

# Mock LLM that calls the filesystem tool
mock_llm = MockLLMWithTools(
    tool_name='mcp_filesystem_read',
    tool_args={'file_path': '/tmp/report.md'},
    final_answer='The monthly report shows revenue of Rs 45,00,000 with 12% growth. Strong performance!'
)

agent = build_mcp_agent(mock_llm, mock_mode=True)

result = agent.invoke({'messages': [HumanMessage('Read the monthly report and summarise it')]})

print('MCP Agent Result:')
print(f'  Total messages: {len(result["messages"])}')
print()
for msg in result['messages']:
    msg_type = type(msg).__name__
    content = msg.content[:80] if msg.content else ''
    tool_calls = getattr(msg, 'tool_calls', [])
    print(f'  [{msg_type}] {content}')
    if tool_calls:
        print(f'           tool_calls: {[tc["name"] for tc in tool_calls]}')

## 5. MCP vs Direct Integration Comparison

In [ ]:
comparison = [
    ('Dimension',        'Direct Integration',              'MCP'),
    ('Setup',            'Write SDK-specific code',         'npx @modelcontextprotocol/server-*'),
    ('Discovery',        'Hardcoded tool list',             'Dynamic at runtime'),
    ('Standards',        'Each tool has different API',     'Unified interface'),
    ('Maintainability',  'Update code for each API change', 'Update server, agent unchanged'),
    ('Security',         'Credentials in your code',        'Credentials in server config'),
    ('Reusability',      'Tool tied to your agent',         'Server usable by any MCP client'),
]

print(f'{comparison[0][0]:<18} {comparison[0][1]:<35} {comparison[0][2]}')
print('-' * 80)
for row in comparison[1:]:
    print(f'{row[0]:<18} {row[1]:<35} {row[2]}')

## Databricks Bridge — MCP + Databricks

In [ ]:
# Databricks MCP Server: databricks-labs/mcp-server-databricks
# Lets your AI agent:
#   - Run SQL queries on Unity Catalog
#   - Trigger Databricks Jobs
#   - Read/write notebooks
#   - Execute MLflow experiments
#
# databricks_mcp_config = {
#     'databricks': {
#         'command': 'uvx',
#         'args': ['databricks-mcp'],
#         'env': {
#             'DATABRICKS_HOST':  os.environ['DATABRICKS_HOST'],
#             'DATABRICKS_TOKEN': os.environ['DATABRICKS_TOKEN'],
#         },
#         'transport': 'stdio',
#     }
# }
#
# async with MultiServerMCPClient(databricks_mcp_config) as client:
#     tools = client.get_tools()  # discovers run_job, query_sql, etc.
#     agent = create_react_agent(llm, tools)
#     result = await agent.ainvoke({
#         'messages': [HumanMessage(
#             'Query the sales table and run the monthly_report_job job'
#         )]
#     })

print('Databricks MCP server: your agent can query Unity Catalog and trigger jobs!')
print('Install: pip install databricks-mcp')
print('Docs: https://github.com/databricks-labs/mcp-server-databricks')